# Chat with Your Emails — Pipeline

Run each cell top to bottom. Every cell is independent — stop/resume whenever.

**Kernel:** `chat-emails`

**Prerequisites:**
1. `data/credentials.json` — download from [Google Cloud Console](https://console.cloud.google.com) → APIs & Services → Credentials → OAuth 2.0 Client ID → Desktop app
2. Add your Gmail as test user: APIs & Services → OAuth consent screen → Test users → Add users
3. Ollama running on `192.168.0.250:11434` with models pulled
4. Qdrant running on `192.168.0.250:6333`

**CLI alternatives** are shown in each cell's markdown header.

## 0. Setup

No CLI equivalent — just imports.

In [ ]:
import os, sys, json
from datetime import datetime
from tqdm import tqdm
import ollama

os.chdir('/home/rapids/chat-with-your-emails')
sys.path.insert(0, '.')

from config.settings import config
from src.tracking.state import PipelineStateManager, PipelineStage

print(f'Project: {os.getcwd()}')
print(f'Ollama:  {config.ollama.base_url}')
print(f'Qdrant:  {config.qdrant.host}:{config.qdrant.port}')
print(f'Models:  LLM={config.models.text_llm} | VLM={config.models.vision_llm} | Embed={config.models.embedding} | Chat={config.models.chat}')

## 1. Verify Connections

Check Ollama and Qdrant are reachable and models exist.

**CLI:** Just read the output of this cell.

In [ ]:
from qdrant_client import QdrantClient

# Ollama
try:
    oc = ollama.Client(host=config.ollama.base_url)
    models = [m['name'] for m in oc.list()['models']]
    print(f'Ollama: OK — {len(models)} models')
    for m in [config.models.text_llm, config.models.vision_llm, config.models.embedding, config.models.chat]:
        hit = any(m in x for x in models)
        print(f'  {m}: {"✓" if hit else "✗ MISSING — run: ollama pull " + m}')
except Exception as e:
    print(f'Ollama: FAILED — {e}')

# Qdrant
try:
    qc = QdrantClient(host=config.qdrant.host, port=config.qdrant.port)
    cols = qc.get_collections().collections
    print(f'Qdrant: OK — {len(cols)} collections')
    for c in cols:
        info = qc.get_collection(c.name)
        print(f'  {c.name}: {info.points_count} points')
except Exception as e:
    print(f'Qdrant: FAILED — {e}')

## 2. Gmail OAuth (first run only)

Opens a browser for Google consent. After first run, `data/token.json` is saved and you never see this again.

**CLI:** `email-ingest --limit 1` (triggers auth automatically)

In [ ]:
from src.ingestion.gmail_client import GmailClient

if not os.path.exists(config.gmail.credentials_file):
    print(f'MISSING: {config.gmail.credentials_file}')
    print('Download from Google Cloud Console → APIs & Services → Credentials → OAuth 2.0')
else:
    print(f'Credentials: {config.gmail.credentials_file} ✓')

if os.path.exists(config.gmail.token_file):
    print(f'Token: {config.gmail.token_file} ✓ (already authenticated)')
else:
    print(f'Token: not found — will open browser for OAuth consent')
    gmail = GmailClient()
    print('Authenticated! Token saved.')

## 3. Ingest Emails

Fetch from Gmail and save to `data/raw_emails/`. Skips already-fetched emails.

**CLI:** `email-ingest --limit 10`

In [ ]:
from src.ingestion.gmail_client import GmailClient, save_email

LIMIT = 10  # How many emails to fetch
QUERY = ''  # Gmail search filter, e.g. 'from:bank' or 'subject:invoice' or 'is:unread'

gmail = GmailClient()
emails = gmail.fetch_emails(max_results=LIMIT, query=QUERY)
print(f'Fetched {len(emails)} emails from Gmail')

# Dedup
raw_dir = 'data/raw_emails'
existing = set()
if os.path.exists(raw_dir):
    existing = {f.replace('.json', '') for f in os.listdir(raw_dir) if f.endswith('.json')}

new_emails = [e for e in emails if e['message_id'] not in existing]
print(f'New: {len(new_emails)} | Already fetched: {len(emails) - len(new_emails)}')

# Save + register in state
state = PipelineStateManager()
subjects = {e['message_id']: e.get('subject', '') for e in new_emails}
senders = {e['message_id']: e.get('sender', '') for e in new_emails}
state.register_emails([e['message_id'] for e in new_emails], subjects=subjects, senders=senders)

for email in tqdm(new_emails, desc='Saving', unit='email'):
    save_email(email)
    state.set_stage(email['message_id'], PipelineStage.FETCHED)

print(f'\nSaved {len(new_emails)} emails to {raw_dir}/')
for e in new_emails[:5]:
    print(f"  {e['subject'][:60]} — {e['sender'][:40]}")
if len(new_emails) > 5:
    print(f'  ... and {len(new_emails) - 5} more')

## 4. Preprocess (LLM + VLM extraction)

Runs qwen3:8b for structured extraction + qwen2.5vl:7b for image attachments.
Each email is saved to `data/processed/` immediately (crash-safe).

This is the slow step — ~30-60 seconds per email depending on attachments.

**CLI:** `email-preprocess --limit 10`

In [ ]:
from src.preprocessing.pipeline import PreprocessingPipeline, _load_raw_emails

LIMIT = 10  # How many to preprocess (0 = all fetched)

emails = _load_raw_emails()
print(f'{len(emails)} raw emails available')

state = PipelineStateManager()
pipeline = PreprocessingPipeline(state_manager=state)

processed = pipeline.run_preprocess(emails, limit=LIMIT)

## 5. Inspect Preprocessed Emails

Look at what the LLM extracted. Useful for debugging and verifying quality.

**CLI:** `ls data/processed/` and `cat data/processed/<id>.json | python3 -m json.tool`

In [ ]:
# Pick an email to inspect
processed_dir = 'data/processed'
files = sorted(os.listdir(processed_dir)) if os.path.exists(processed_dir) else []
print(f'{len(files)} processed emails')

if files:
    # Show the first one
    with open(os.path.join(processed_dir, files[0])) as f:
        data = json.load(f)
    
    print(f"\n{'='*60}")
    print(f"Subject: {data['subject']}")
    print(f"From:    {data['sender']}")
    print(f"Date:    {data['date']}")
    print(f"Category: {data['category']} / {data.get('subcategory', '')}")
    print(f"Sentiment: {data['sentiment']} | Tone: {data.get('tone', '')}")
    print(f"Important: {data['is_important']} | Needs response: {data['requires_response']}")
    print(f"\nSummary:\n{data['summary']}")
    
    if data.get('key_information'):
        print(f"\nKey information:")
        for info in data['key_information'][:5]:
            print(f"  • {info}")
    
    if data.get('action_items'):
        print(f"\nAction items:")
        for item in data['action_items'][:3]:
            if isinstance(item, dict):
                print(f"  • {item.get('task', '')} (priority: {item.get('priority', '')})")
            else:
                print(f"  • {item}")
    
    entities = data.get('entities', {})
    if entities and isinstance(entities, dict):
        for key, vals in entities.items():
            if vals:
                print(f"\n{key}: {', '.join(str(v) for v in vals[:5])}")
    
    print(f"\nNoise ratio: {data.get('noise_ratio', 0):.0%}")
    print(f"Attachments: {len(data.get('attachment_descriptions', []))} described, {len(data.get('attachment_skipped', []))} skipped")
    print(f"Chunks: {len(data.get('chunks', []))}")

## 6. Embed & Store

Chunks preprocessed emails, embeds with bge-m3, stores in Qdrant.
Chat works immediately after this cell.

**CLI:** `email-embed --limit 10`

In [ ]:
from src.preprocessing.pipeline import PreprocessingPipeline

LIMIT = 10  # How many to embed (0 = all preprocessed)

state = PipelineStateManager()
pipeline = PreprocessingPipeline(state_manager=state)

embedded = pipeline.run_embed(limit=LIMIT)
print(f'\nEmbedded {embedded} emails into Qdrant')

## 7. Chat!

Ask questions about your emails. Change `QUESTION` and re-run.

**CLI:** `email-chat` then open `http://localhost:8501/docs`

In [ ]:
from src.embedding.embedder import EmailEmbedder
from src.storage.vector_store import EmailVectorStore

embedder = EmailEmbedder()
store = EmailVectorStore()
llm_client = ollama.Client(host=config.ollama.base_url)

info = store.get_collection_info()
print(f'Store: {info["points_count"]} chunks | Chat model: {config.models.chat}')

In [ ]:
QUESTION = "What are the most important emails I received recently? Summarize them."  # <-- CHANGE THIS

SYSTEM = """You are a helpful email assistant. Answer based on the provided email context.
Be concise. Reference sender, date, and subject. If context is insufficient, say so."""

# Retrieve
query_emb = embedder.embed_text(QUESTION)
results = store.search(query_emb, limit=5)

context = '\n\n---\n\n'.join(
    f"[Email {i+1}] From: {r['sender']} | Date: {r['date']} | Subject: {r['subject']}\n"
    f"Category: {r['category']}\n{r['text'][:1500]}"
    for i, r in enumerate(results)
)

# Generate
response = llm_client.chat(
    model=config.models.chat,
    messages=[
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Email context:\n\n{context}\n\nQuestion: {QUESTION}'},
    ],
    options={'temperature': 0.3},
)

print(f'Q: {QUESTION}')
print('=' * 60)
print(response['message']['content'])
print('\nSources:')
for r in results:
    print(f"  [{r['score']:.3f}] {r['subject'][:50]} — {r['sender'][:30]}")

## 8. Pipeline Status

Check progress, errors, and what's left.

**CLI:** `email-status` or `email-errors`

In [ ]:
state = PipelineStateManager()
progress = state.get_progress()

print(f"Status: {progress['status']}")
print(f"Total:  {progress['total_emails']} emails")
print(f"Overall: {progress['overall_pct']}%")
print()

stages = progress['stages']
for stage, label in [('fetched','Fetched'), ('cleaned','Cleaned'), ('llm_extracted','LLM'), 
                      ('vlm_processed','VLM'), ('embedded','Embedded'), ('stored','Stored')]:
    s = stages.get(stage, {'completed': 0, 'failed': 0})
    bar = '█' * (s['completed'] * 20 // max(progress['total_emails'], 1)) + '░' * 20
    print(f"  {label:12} {s['completed']:4} done  {s['failed']:4} fail  {bar}")

errors = state.get_failed_emails()
if errors:
    print(f"\n{len(errors)} errors:")
    for e in errors:
        print(f"  [{e['stage']}] {e['subject'][:40]} — {e['error'][:60]}")

## 9. Reset (nuclear option)

Clears all pipeline state. Qdrant data is NOT deleted — use `email-reset` for that.

**CLI:** `email-reset`

In [ ]:
# Uncomment to reset:
# state = PipelineStateManager()
# state.reset()
# print('Pipeline state cleared. Re-run from step 3.')